<a href="https://colab.research.google.com/github/anandharidas/KernelProgramming/blob/main/Type_4_Matrix_mutiplier_using_1D_Tiling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
from numba import cuda, float32
import cupy as cp
import numpy as np

BM = 32
BN = 32
BK = 32
TM = 8  # no. of output elements process in single thread
NT = (BM * BN) // TM

ceild_div = lambda a, b: (a + b - 1) // b

@cuda.jit
def matmul( matA, matB, matC, m, n, k) :
  tileA = cuda.shared.array(shape=(BM,BK) , dtype = cp.float32)
  tileB = cuda.shared.array(shape=(BK,BN) , dtype = cp.float32)

  tid = cuda.threadIdx.x
  col = tid % BN
  row = (tid // BN) * TM

  block_row = cuda.blockIdx.y * BM
  block_col = cuda.blockIdx.x * BN

  # register allocation and initialization
  acc = cuda.local.array(TM, dtype=cp.float32)
  for t in range(TM):
    acc[t] = float32(0.0)

  for k0 in range(0,k,BK):
    # phase 1:
    # threads will load values from matA and matB
    # into tileA and tileB
    # gmem -> smem

    # phase 1: load values from matA into tileA
    for i in range(tid, BM * BK, NT):
      # convert linear coordinates into 2D coordinates (r,c)
      r = i // BK
      c = i % BK
      global_mem_row = block_row + r
      global_mem_col = k0 + c
      if global_mem_row < m and global_mem_col < k:
        tileA[r,c] = matA[global_mem_row, global_mem_col]
      else:
        tileA[r,c] = float32(0.0)

    # phase 1: load values from matB into tileB
    for i in range(tid, BK * BN, NT):
      r = i // BN
      c = i % BN
      global_mem_row = k0 + r
      global_mem_col = c + block_col
      if global_mem_row < k and global_mem_col < n:
        tileB[r,c] = matB[global_mem_row, global_mem_col]
      else:
        tileB[r,c] = float32(0.0)

    # sync threads
    cuda.syncthreads()

    # phase 2: load values from matB into tileB
    for kk in range(BK):
      b = tileB[kk,col]
      for t in range(TM):
        acc[i] += tileA[row + t, kk] * b

    # sync threads
    cuda.syncthreads()

  global_mem_col = block_col + col
  for t in range(TM):
    global_mem_row = block_row + row + t
    if global_mem_row < m and global_mem_col < n:
      matC[global_mem_row, global_mem_col] = acc[0]


m = 1000
k = 1200
n = 800

matA = cp.ones(shape=(m,k), dtype=cp.float32) * 2.0
matB = cp.ones(shape=(k,n), dtype=cp.float32) * 3.0
# Fixed typo: changed cp.zero to cp.zeros
matC = cp.zeros(shape=(m,n), dtype=cp.float32)

threads_per_block = (32,32)
blocks_per_grid = (ceil_div(n, threads_per_block[0]), ceil_div(m, threads_per_block[1]))
matmul[blocks_per_grid,threads_per_block](matA, matB, matC, m, n, k)

cuda.synchronize()

matA = cp.asnumpy(matA)
matB = cp.asnumpy(matB)
# Fixed typo: changed cp.asumpy to cp.asnumpy
matC = cp.asnumpy(matC)

np.testing.assert_allclose(matC, matA @ matB, rtol=1e-5, atol=1e-5)



TypingError: Failed in cuda mode pipeline (step: nopython frontend)
NameError: name 'product' is not defined
During: Pass nopython_type_inference